# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YoussefZaky208/Flyrank-ML-Track-Assignemnt/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2 — Refresh / Content Opportunity Scoring.**

I'm picking this lane because it already has the strongest evidence in the starter data that a
learned ranking beats a plain rule (see Section 3), and it produces the most direct kind of
output for this internship: a ranked queue that tells a content reviewer which page to look at
first. The other lanes are interesting (Lane 1 signal analysis, Lane 4 CTR scoring, Lane 3
clustering) but none of them end in an action queue by default the way Lane 2 does, and Lane 2
is the one the starter pipeline (`scripts/01`-`05`) was built to demonstrate end-to-end — so I
can lean on `outputs/model_report.md` and `outputs/refresh_queue_sample.csv` as a working
reference while I build my own, stronger version (future-window label instead of the
current-window proxy label the starter uses).

In [ ]:
# Section 1 is a qualitative choice (lane pick) - no supporting code required here.
# See Section 3 below for the real numbers backing this choice.


## 2. The question: decision, action, cost of a wrong call

**Search question:** Which content pages should a content reviewer look at first for a refresh,
expansion, or protection decision, given limited weekly review capacity?

**Unit of analysis:** one content item (`content_id`) — a single pseudonymized page, using its
trailing 90-day metrics as of the snapshot date. (In a future-window version, the unit would be
"a content item at a given decision point in time.")

**Output:** a ranked review queue — every eligible content item gets a priority score, a reason
code (why it's on the list), and a suggested action (refresh / expand / protect / monitor).

**Decision it improves:** which pages a content/SEO reviewer opens first when they only have
time to review a fraction of the site this week.

**Who acts, and what they do:** a content strategist or SEO reviewer with fixed weekly capacity
(say, 20-50 pages) reads the top of the queue, checks the reason code against the real page, and
decides whether to refresh the content, expand it, leave it alone, or flag it for a bigger
rewrite.

**Cost of a wrong call:**
- *False positive* (queue says "review this" but the page was actually fine): wastes a
  reviewer's limited time — time that could have gone to a page that really needed it. With
  capacity fixed, every wasted slot is a real opportunity cost, not just an inconvenience.
- *False negative* (a genuinely declining, high-demand page never surfaces near the top):
  the page keeps losing visibility/clicks silently until someone notices by accident, which is
  the more expensive failure since it compounds over time.
- Because reviewer time is the scarce resource, precision in the top of the queue (precision@K)
  matters more here than overall accuracy — a queue that's noisy at rank 500 is fine; a queue
  that's noisy at rank 20 wastes the reviewer's actual working set.

**Why data/ML can help at all:** the "worth reviewing" signal is not one clean rule — it's a mix
of demand (impressions), movement (trend), staleness (age/freshness), and depth (word count),
and these interact (e.g., a stale page only matters if it still has demand). A single if-statement
rule can approximate this (and the starter baseline does), but the starter pipeline's own
results show a random forest ranks the same rows much better than the hand-written rule
(precision@50 of 0.740 vs. 0.240 — see Section 3), which is exactly the situation where a model
earns its place over a plain rule: the pattern is real but too tangled for a human to hand-tune.

In [ ]:
# Section 2 is the framing paragraph - no code required here.
# See Section 3 below for the numbers that motivate the decision/action/cost story above.


## 3. Quick look at the data (2-3 real numbers)

Numbers below are computed live from `data/raw/content_refresh_anonymized.csv` (30,000 rows, 32
clients) and cross-checked against `outputs/model_report.md` for the pipeline's own results.

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Rows: {len(df):,}  |  Clients: {df['client_id'].nunique()}")

# Number 1: how big is the "declining with demand" candidate pool?
declining_with_demand = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]
pct_declining = len(declining_with_demand) / len(df) * 100
print(f"declining_with_demand rows: {len(declining_with_demand):,} "
      f"({pct_declining:.1f}% of all rows) -- a large, well-populated candidate pool")

# Number 2: how big is the stricter "stale visible page" candidate pool?
stale_visible = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]
print(f"stale_visible_page rows: {len(stale_visible):,} -- a much stricter, smaller pool, "
      "showing the queue needs several reason codes, not just one rule")

# Number 3: does a learned model actually beat the hand-written baseline rule on this data?
# (pulled from the starter pipeline's own verified output, not re-derived here)
with open("outputs/model_report.md") as f:
    report = f.read()
print()
print("From outputs/model_report.md (starter pipeline, already run and committed):")
in_table = False
for line in report.splitlines():
    if line.startswith("| Model"):
        in_table = True
    if in_table:
        print(" ", line.strip())
        if in_table and line.strip() == "":
            break


Rows: 30,000  |  Clients: 32
declining_with_demand rows: 13,152 (43.8% of all rows) -- a large, well-populated candidate pool
stale_visible_page rows: 17 -- a much stricter, smaller pool, showing the queue needs several reason codes, not just one rule

From outputs/model_report.md (starter pipeline, already run and committed):
  | Model | ROC AUC | Avg precision | Precision@50 | Recall | F1 |
  |---|---:|---:|---:|---:|---:|
  | decision_tree | 0.742 | 0.575 | 0.540 | 0.716 | 0.634 |
  | logistic_regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
  | random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
  | baseline_rules | 0.627 | 0.468 | 0.240 | - | - |
  


## 4. Careful words: what I can and can't claim

**What this work will be able to say:**
- *Observed*: which pages, in this 90-day trailing window, show the combination of demand,
  decline, staleness, or thinness that the data contract defines as "worth reviewing."
- *Directional*: that a learned ranking, validated on held-out clients, orders review candidates
  better than a simple hand-written rule, by a stated metric (precision@K) on this dataset.
- *Decision-support*: a ranked queue with reason codes that a human reviewer still has to open,
  read, and judge before acting. The model proposes; it does not decide.

**What this work will never claim:**
- That refreshing a page *causes* it to recover — that requires a real experiment (e.g., an
  A/B test or before/after design with a control group), not an observational ranking.
- That any result reflects how Google's (or any AI platform's) ranking algorithm actually works.
  I only have FlyRank's observed search/engagement signals, not the algorithm itself.
- That the current-window proxy label (`trend_direction == "down"`, used by the starter
  pipeline) is the same as a true future outcome. It's a beginner proxy label calculated from
  the same window as the features, not a forward-looking one. If I move to a future-window label
  (prior 90 days -> next 30 days decline) later, I'll say so explicitly and re-run the leakage
  checklist, since that's a materially stronger and different claim than the proxy label.
- That results from the 30,000-row starter slice automatically hold at the ~79M-row warehouse
  scale — that has to be re-earned with its own validation, not assumed.

In [ ]:
# Section 4 is a written commitment about claim scope - no code required here.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.